# 03 · 内置提示词与请求装配

同样是「确认起息日是否算第一天」，Build 可以继续改代码，Explore 被要求专注搜索和分析，Plan 会把这次对话当成方案讨论。要解释这三种行为，先分清 **角色指令、项目事实、工具接口、执行权限** 各自由谁提供。比如 Explore 的角色指令要求只读，工具层还会按权限筛选和检查实际调用。前者让模型理解任务边界，后者负责执行约束。[内置 Agent](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/agent.ts#L54) · [按权限生成工具快照](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/tool.ts#L312)



## 从当前环境到模型请求

1. **选出这一步的 Agent、模型与工具。** `SessionContext.select()` 合并 Agent 和 Session 权限，再依据模型策略生成工具快照。它同时收集 Code Mode 目录、MCP、References、Skills、机器、项目指令、日期和环境等来源。来源顺序优先排列更容易在多个会话间保持相同的内容，有利于复用提示词缓存前缀。[会话上下文选择](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/context.ts#L148)
2. **把会话已有内容装配成请求草稿。** `baseTranscript()` 用 `agent.system` 或默认模板形成首个 system 段，再加入已保存的指令基线。历史里的用户消息、助手回复和工具结果分别转换为模型消息。项目文件中的说明是运行时提供的指令来源，仓库随附模板则负责定义通用行为。[请求草稿](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/model-request.ts#L111) · [读取历史与基线](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/history.ts#L290)
3. **按用途运行 Hook。** 主循环、压缩、辅助生成、命名分别进入 `context`、`compaction`、`generate`、`title` 钩子。Hook 可以修改 system、消息和工具定义，因此同一个工具的静态说明、快照里的说明和最后发给模型的说明需要分别查证。[请求用途与 Hook](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/model-request.ts#L468)
4. **生成协议请求并执行。** 主请求用 Hook 返回的 system、消息和工具定义构造，删掉只用于本地 Hook 的消息元数据并应用模型输入能力限制。工具仍按注册身份映射回本地执行器。排查模型行为时，以当次请求的数据为依据。[工具映射与实际请求](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/model-request.ts#L286)

## 系统、角色和模型模板

| 内置项 | 内容与注入条件 | 源码 |
|---|---|---|
| 通用系统模板 | 当 Agent 没有显式 `system` 时用作首个 system 段。根据直出工具名插入 `shell`、`write`、`edit` 的使用说明。 | [模板](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/runner/prompt/system.txt) · [渲染器](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/system-prompt.ts#L3) |
| Build | 默认主 Agent。核心注册在此处设置角色描述和权限，普通请求走通用系统提示词路径。 | [角色配置](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/agent.ts#L56) |
| General | 通用子 Agent。该处设置的简短职责也可随 `subagent` 工具描述提供给父 Agent，帮助它决定派发给谁。 | [角色配置](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/agent.ts#L63) · [派发说明动态拼装](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/tool/plugin/subagent.ts#L240) |
| Explore | 专门搜索与分析。注册时设置独立的角色 system，另行配置工具权限。 | [角色提示词](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/agent.ts#L7) · [角色与权限](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/agent.ts#L74) |
| Plan | 进入和退出 Plan 时，向会话注入对应提醒。若压缩后历史缺少有效提醒，`context` Hook 可在消息尾部附近补齐。编辑限制另外由权限规则落实。 | [Plan 状态提示](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/plan.ts#L12) · [请求中补齐提醒](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/plan.ts#L55) |
| OpenAI 系列适配 | ID 包含 `gpt` 时替换首段。ID 同时包含 `astra` 时使用 Astra 专用版本。 | [GPT 模板](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/system-prompt/gpt.txt) · [Astra 模板](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/system-prompt/gpt-astra.txt) · [匹配条件](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/optimize.ts#L16) |
| Claude 系列适配 | ID 包含 `claude` 时追加至首段。 | [Claude 模板](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/system-prompt/anthropic.txt) · [追加条件](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/optimize.ts#L22) |
| Kimi、Trinity、Muse 系列适配 | 分别按 ID 子串替换首段。Muse 模板还填入显示模型名。 | [Kimi](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/system-prompt/kimi.txt) · [Trinity](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/system-prompt/trinity.txt) · [Muse](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/system-prompt/meta.txt) · [匹配条件](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/optimize.ts#L32) |
| 模型身份 | 在相关请求的 system 块中插入模型名、Provider ID 和 Model ID。 | [身份 Hook](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/identity.ts#L10) |
| Claude Code 提供方的运行时模板 | 走 Claude Agent SDK 时，主请求和派生请求采用相应的 Claude Code 预设，可附加 OpenCode 角色和指令。独立用途请求另有辅助 system 路径。 | [提供方系统模板](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/provider/claude-agent/language-model/system-prompt.ts#L9) · [按用途选择](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/provider/claude-agent/language-model/claude-agent-language-model.ts#L45) |

模型模板的追加或替换发生在首个 system 段。若 Agent 已明确设置自己的 `system`，优化插件会跳过改写。独立的项目指令块和其他 Hook 仍按各自逻辑处理。比如 Explore 对同一个 GPT 模型保留它的探索角色，项目里的测试规范依然可以随指令基线进入请求。[模板 Hook 的守卫与改写位置](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/optimize.ts#L59)

## 工具说明和环境指令

| 内容来源 | 模型怎样看见 | 源码 |
|---|---|---|
| 项目、全局和配置指令 | 发现 `AGENTS.md`、启用后可用的 `CLAUDE.md` 回退，以及配置额外指定的来源，作为指令基线或变化进入会话。读取带有更具体指令的路径时，还可按需加载并记录到会话。 | [默认发现规则](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/instruction-discovery.ts#L128) · [配置和来源合并](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/config/plugin/instruction.ts#L192) · [按需注入](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/instructions.ts#L47) |
| 日期、环境、工作位置 | 根据 Location 和运行时状态渲染日期、工作目录、工作区根目录、平台等信息。 | [动态内置指令](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/instructions/builtins.ts#L21) |
| 工具描述、参数和格式 | 每个工具随定义提供描述及输入 Schema。比如 `read` 的参数带路径和行范围说明，`patch` 使用补丁格式提示，`todowrite` 带随仓使用指南。Hook 还可在请求前完善说明。 | [定义生成](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/tool/runtime.ts#L27) · [读取工具](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/tool/plugin/read.ts#L22) · [补丁工具](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/tool/plugin/patch.ts#L18) · [待办提示](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/tool/todowrite.txt) |
| 动态工具说明 | `subagent` 可附带适用的子 Agent 名称和职责。Shell 说明会按所选 Shell 渲染。工作树相关提示按可调用入口提示直接调用或 Code Mode 路径。 | [子 Agent](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/tool/plugin/subagent.ts#L240) · [Shell](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/tool/plugin/shell.ts#L560) · [工作树指导](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/tool/plugin/opencode.ts#L97) |
| Code Mode | 存在可用目录时渲染调用说明和工具目录。目录变化时可生成更新说明。 | [目录提示渲染](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/codemode/instructions.ts#L9) · [指令来源](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/codemode/instructions.ts#L147) |
| Skills、References、MCP、机器 | Skills 和 References 可以先展示描述供模型选择，MCP 指令会按当前快照中相关工具的可见性筛选，机器信息按功能与当前环境加载。随仓的 OpenCode 和 Report Skill 另有各自的内容。 | [Skills](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/skill/instructions.ts#L55) · [内置 Skill 注册](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/skill.ts#L23) · [References](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/reference/instructions.ts#L51) · [MCP](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/mcp/instructions.ts#L64) · [机器](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/machine/instructions.ts#L36) |

指令来源有稳定的键。首次装配建立基线，之后检测到内容变化时，将增量和渲染文本写入事件。于是跨 Step 重建请求时，模型仍能读懂「哪条规则后来变了」。这也是理解提示词缓存的一个抓手。稳定的前缀和变化发生时间都能从会话数据看见。[指令键和读取接口](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/instructions/index.ts#L28) · [基线与变化事件](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/instruction-state.ts#L38)

## 有目标的专用提示词

| 入口 | 用途和位置 | 源码 |
|---|---|---|
| 标题 | 隐藏 Title Agent 的 system 要求输出简短会话名。命名请求另行发送，用户输入提供待命名素材。 | [Title 指令](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/agent.ts#L19) · [命名请求](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/title.ts#L74) |
| Summary | 内置的隐藏 Summary Agent 有自己的简短会话总结指令。历史压缩另有结构化 checkpoint 模板。 | [Summary 指令](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/agent.ts#L40) · [注册](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/agent.ts#L110) |
| 上下文压缩 | `buildPrompt()` 按新建或更新 checkpoint 选择模板，保留用户需求、决定和待办。旧格式还有专用迁移提醒，未按模板输出时可提示重做。请求提取当前对话上下文进行总结。 | [压缩模板与纠偏](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/compaction.ts#L211) · [压缩请求准备](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/compaction.ts#L843) |
| `/goal` | 启动时展开目标模板。会话即将空闲时，通过单独的生成请求请模型判定状态。若需继续，注入下一步。目标因压缩而不可见时，Hook 临时补提醒。 | [启动模板](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/goal/kickoff.txt) · [判定模板](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/goal/judge.txt) · [继续模板](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/goal/continuation.txt) · [提醒模板](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/goal/reminder.txt) · [判定与继续](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/goal.ts#L104) |
| `/init`、`/review` | 展开工作区路径及命令参数，通过 `session.prompt()` 作为命令输入交给会话。 | [初始化模板](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/command/initialize.txt) · [审查模板](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/command/review.txt) · [命令分派](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/command.ts#L27) |
| `/automation` | 启用自动化后注册命令，将参数填进自动化任务的专门模板，再投递给会话。 | [自动化模板](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/command/automation.txt) · [命令注册](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/tool/plugin/automation.ts#L159) |
| `/btw` | 从 TUI 发起一次临时问答，指令要求用当前上下文直接回答。调用 `session.generate`，结果显示在侧问对话框。 | [临时问答指令与入口](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/tui/src/feature-plugins/prompt/btw.tsx#L16) |
| Step 上限 | 到达 Agent 的 Step 上限时，在消息末尾插入文字提示，同时把 `toolChoice` 设为 `none`。定义仍可留在请求里以复用模型缓存前缀。 | [上限提示](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/runner/max-steps.ts#L1) · [上限执行策略](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/runner/llm.ts#L261) |
| 会话分叉提示 | Fork Hook 将后台工作归属、共享文件和后续协作的说明加入分支会话。 | [分叉提示与投递](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/fork-notice.ts#L7) |
| Home Memory | 在启用且当前 Location 为家目录时，注册记忆参考资源，并可将维护提示投递给单独的会话。 | [提示与启用条件](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/home-memory.ts#L34) |
| 缓存保活 | 在配置启用且时间窗口适合时，使用短辅助提示触发模型保活请求。 | [保活提示与启用条件](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/plugin/warming.ts#L13) |

## 观察实验 · 同一任务换角色

固定任务「查明订单的服务期限从哪一天开始」，对照 Build 和 Explore。两组请求都提供同一份项目规范和同一条用户问题。观察 **第一个 system 块的来源、项目规范的位置、实际工具定义、是否触发模型模板 Hook**。分析时先把模板和 `messages` 分开，再核对 `tools` 与执行权限。

- **Build + GPT 模型**。预计通用首段可被 GPT 模板替换，项目规范仍经指令基线进入请求。
- **Explore + 同一 GPT 模型**。预计 Explore 自带 system 保留，工具定义按其角色和会话权限筛选。
- **Build 到达 Step 上限**。预计工具名仍可能出现在请求字段里，但 `toolChoice=none` 会阻止本步继续选工具。

自测时回答两个问题。把新团队规范写进项目指令后，已经进行多步的会话怎样看到变化？如果要保证某个 Agent 无法编辑文件，除了角色提示词，还需要检查哪条工具授权？这两个问题分别追到指令事件，以及实际工具请求和授权链路。[指令变化](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/instruction-state.ts#L38) · [工具执行核验](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/tool.ts#L176)

In [1]:
from pathlib import Path
import json

# 从课程根目录或 notebooks/ 打开时均可读取随附资料。
HERE = Path.cwd().resolve()
COURSE = next(folder for folder in (HERE, *HERE.parents) if (folder / "fixtures" / "iso-9001.json").is_file())

## 观察实验 · 原版 TypeScript 如何生成工具指导

用 Bun 直接调用附带的原版 `SessionSystemPrompt.make()`。比较只有 `read` 和增加 `shell/write/edit` 两份工具名单。这个实验观察默认模板中的动态工具指导，Agent 专用模板和模型 Hook 见上文。

In [2]:
import difflib
import shutil
import subprocess

bun = shutil.which("bun")
if bun is None:
    print("当前内核未发现 Bun。可先读上文及原版 ./fixtures/upstream/system-prompt.ts。")
else:
    expression = '''
      import { SessionSystemPrompt } from "./fixtures/upstream/system-prompt.ts";
      console.log(JSON.stringify([
        SessionSystemPrompt.make(["read"]),
        SessionSystemPrompt.make(["read", "shell", "write", "edit"])
      ]));
    '''
    output = subprocess.check_output(
        [bun, "--loader", ".txt:text", "-e", expression], cwd=COURSE, text=True
    )
    read_only, with_editing = json.loads(output)
    delta = list(difflib.unified_diff(
        read_only.splitlines(), with_editing.splitlines(),
        fromfile="read", tofile="read + shell + write + edit", n=1, lineterm=""
    ))
    print("\n".join(delta))
    assert "Prefer dedicated tools over shell" in with_editing
    assert "Prefer dedicated tools over shell" not in read_only

--- read
+++ read + shell + write + edit
@@ -7,3 +7,6 @@
 - Prefer parallelizing independent tool calls.
-
+- Prefer dedicated tools over shell commands; fall back to the shell when a tool cannot do what you need.
+- Do not chain shell commands with separators like `echo "====";` or `printf '---'`; the output becomes noisy in a way that makes the user's side of the conversation worse.
+- Use the write tool to create files or completely replace their content. Prefer using the edit tool for targeted changes.
+- Use the edit tool for targeted changes to existing text files. It replaces the exact text in `oldString` with `newString`, and the values must differ. By default, `oldString` must occur exactly once. If it occurs multiple times, include more surrounding context to make it unique or set `replaceAll` to true to replace every occurrence.
 
